# Hafta 3: Veri ve AI Baseline (Colab)

[![Colab'da aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leylagns/avm-kiosk-assistant/blob/main/notebooks/02_colab_week3.ipynb)

Bu not defterinde sırayla şunları yapacağız:

1. Repoyu Colab'a alıp **veri setini kendimiz üreteceğiz**.
2. Veriye bakacağız (kaç örnek, hangi niyetler, zor test neden zor).
3. **E1** (klasik baseline) ve **E2** (BERTurk) modellerini eğitip **aynı testlerde** karşılaştıracağız.
4. Hataları inceleyip sonuçları kaydedeceğiz.

**Önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç. (E1 GPU istemez, E2 ister.)
Hücreleri **yukarıdan aşağıya sırayla** çalıştır. Her hücrenin altındaki "Ne oldu?" notunu oku.

## 1. Repoyu al ve kurulum

In [ ]:
import os
REPO = "https://github.com/leylagns/avm-kiosk-assistant.git"
if not os.path.exists("src/kiosk_nlu"):          # zaten repo içindeysek tekrar klonlama
    if not os.path.exists("avm-kiosk-assistant"):
        !git clone {REPO}
    %cd avm-kiosk-assistant
!git log --oneline | head -5

**Ne oldu?** Kodumuz ve katalog verisi GitHub'dan Colab'ın geçici makinesine indi. Colab'ı kapatınca bu kopya silinir; kalıcı olan GitHub'dır.

In [ ]:
!pip -q install transformers accelerate tabulate
import sys
sys.path.insert(0, "src")   # kiosk_nlu paketini import edebilmek için
import torch
print("GPU var mı:", torch.cuda.is_available())

**Ne oldu?** BERTurk için gereken kütüphaneleri kurduk. Çıktıda `GPU var mı: True` görmelisin. `False` ise çalışma zamanı türünü kontrol et.

## 2. Veri setini üret

Veri setini elle yazmadık; `generate_dataset.py` **şablon + rastgele katalog değerleri + gürültü** ile üretiyor.
`seed=42` sabit olduğu için herkes aynı veriyi elde eder (tekrarlanabilirlik).

In [ ]:
!PYTHONPATH=src python -m kiosk_nlu.generate_dataset

In [ ]:
import pandas as pd
pd.set_option("display.width", 140, "display.max_colwidth", 80)
splits = {n: pd.read_csv(f"data/processed/{n}.csv") for n in ["train", "val", "test", "test_hard"]}
for n, d in splits.items():
    print(f"{n:10s} {len(d):5d} satır")
splits["train"].sample(8, random_state=1)

**Ne oldu?** Dört parça oluştu:
- `train`: modeli eğittiğimiz veri
- `val`: ayar seçmek için (hangi C, kaç epoch gibi)
- `test`: eğitimle aynı tarzda cümleler (kolay)
- `test_hard`: **eğitimde hiç görmediği şablonlar**, örtük ifadeler ("acıktım"), ağır yazım hatası (zor)

Gerçek başarı `test_hard` üzerinde ölçülür.

In [ ]:
print("Niyet dağılımı (train):")
print(splits["train"].intent.value_counts(), "\n")
print("test_hard'dan örnekler:")
splits["test_hard"].sample(10, random_state=3)[["text", "intent", "entity_value"]]

**Soru (kendine sor):** `test_hard` cümleleri `train` cümlelerinden neden farklı yazıldı? Cevap: model şablonları ezberlemiş mi, gerçekten anlıyor mu diye ayırt etmek için.

## 3. E1: klasik baseline (TF-IDF + Lojistik Regresyon)

In [ ]:
from sklearn.metrics import f1_score
from kiosk_nlu.baseline import BaselineNLU
from kiosk_nlu.evaluate import evaluate_split, plot_cm, TARGETS

train, val, test, hard = (splits[k] for k in ["train", "val", "test", "test_hard"])

# C değerini doğrulama setinde seç (test setlerine bakmadan)
best_c, best_f1 = None, -1
for c in [1, 10, 100]:
    m = BaselineNLU(C=c).fit(train.text, train.intent)
    f1 = f1_score(val.intent, m.predict_intents(val.text), average="macro")
    print(f"C={c:<4} val makro-F1={f1:.3f}")
    if f1 > best_f1:
        best_c, best_f1 = c, f1
e1 = BaselineNLU(C=best_c).fit(train.text, train.intent)
print("seçilen C:", best_c)

**Ne oldu?** Cümleyi harf gruplarına (2-5'li karakter parçaları) bölüp sayan basit bir model eğittik. Sözlükten mağaza/olanak/kategori/kat adını bulmak için de bulanık eşleştirme kullanılıyor.
`C` ayarını **sadece `val` ile** seçtik. Test setlerine bakarak ayar seçmek "kopya çekmek" olurdu.

In [ ]:
rows = {}
out_e1_test, rows["E1 test"] = evaluate_split(e1, test)
out_e1_hard, rows["E1 test_hard"] = evaluate_split(e1, hard)
pd.DataFrame(rows).T[["intent_acc", "intent_macro_f1", "entity_acc", "latency_p95_ms"]].round(3)

**Ne oldu?** Metrikler:
- `intent_macro_f1`: niyet başarısı (Q1 hedefi: zor testte ≥ 0.85)
- `entity_acc`: varlığı doğru bulma (Q2 hedefi: zor testte ≥ 0.85)
- `latency_p95_ms`: 100 sorgunun 95'inin yanıt süresi (Q3 hedefi: ≤ 500 ms)

Beklenti: `test`'te çok yüksek, `test_hard`'da belirgin düşük çıkması.

## 4. E2: BERTurk

BERTurk, Türkçe metinlerle önceden eğitilmiş hazır bir modeldir. Sıfırdan eğitmiyoruz; kendi verimizle **ince ayar** yapıyoruz.
İki ayrı baş eğitiyoruz: biri niyeti, biri varlığı (ör. `store:S17`) tahmin ediyor. Eğitim T4 GPU'da birkaç dakika sürer.

In [ ]:
from kiosk_nlu.bert_nlu import BertNLU
e2 = BertNLU(epochs=4).fit(train)

**Ne oldu?** Her epoch'ta `loss` değerinin düşmesi gerekir. Düşmüyorsa bir şeyler ters gidiyordur; bana yaz.

In [ ]:
out_e2_val, m_val = evaluate_split(e2, val)
print("E2 val makro-F1:", round(m_val["intent_macro_f1"], 3), "| varlık:", round(m_val["entity_acc"], 3))

out_e2_test, rows["E2 test"] = evaluate_split(e2, test)
out_e2_hard, rows["E2 test_hard"] = evaluate_split(e2, hard)
pd.DataFrame(rows).T[["intent_acc", "intent_macro_f1", "entity_acc", "latency_p95_ms"]].round(3)

## 5. Karşılaştırma ve hedefler

In [ ]:
cmp = pd.DataFrame(rows).T[["intent_acc", "intent_macro_f1", "entity_acc", "end_to_end_acc", "latency_mean_ms", "latency_p95_ms"]].round(3)
display(cmp)

hard_e1, hard_e2 = rows["E1 test_hard"], rows["E2 test_hard"]
def durum(v, t, alt=False):
    return "✅" if (v <= t if alt else v >= t) else "❌"
hedef = pd.DataFrame({
    "Ölçüt": ["Q1 niyet makro-F1 (hard)", "Q2 varlık doğruluğu (hard)", "Q3 gecikme p95 (ms)"],
    "Hedef": [TARGETS["intent_macro_f1_hard"], TARGETS["entity_acc_hard"], TARGETS["p95_latency_ms"]],
    "E1": [round(hard_e1["intent_macro_f1"], 3), round(hard_e1["entity_acc"], 3), round(hard_e1["latency_p95_ms"], 1)],
    "E2": [round(hard_e2["intent_macro_f1"], 3), round(hard_e2["entity_acc"], 3), round(hard_e2["latency_p95_ms"], 1)],
})
hedef["E1 durum"] = [durum(hard_e1["intent_macro_f1"], hedef.Hedef[0]), durum(hard_e1["entity_acc"], hedef.Hedef[1]), durum(hard_e1["latency_p95_ms"], hedef.Hedef[2], True)]
hedef["E2 durum"] = [durum(hard_e2["intent_macro_f1"], hedef.Hedef[0]), durum(hard_e2["entity_acc"], hedef.Hedef[1]), durum(hard_e2["latency_p95_ms"], hedef.Hedef[2], True)]
hedef

**Ne oldu?** İki yöntemi aynı verilerde, aynı ölçütlerle kıyasladık ve öneri formundaki hedeflerle karşılaştırdık.
Sunumda anlatacağın cümle şuna benzer olmalı: *"Klasik baseline zor testte X, BERTurk Y aldı; hedef 0.85 idi; kalan açık şu hata türlerinden geliyor."*

## 6. Hata analizi

In [ ]:
def hatalar(out):
    bad = out[~(out.intent_ok & (out.entity_ok | out.entity_type.isna()))]
    return bad[["text", "intent", "pred_intent", "entity_value", "pred_entity"]]

e2_err = hatalar(out_e2_hard)
print("E2 hatalı örnek:", len(e2_err), "/", len(out_e2_hard))
e2_err.head(20)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
os.makedirs("docs/week3/figures", exist_ok=True)
plot_cm(out_e1_hard, "docs/week3/figures/confusion_e1_hard.png", "E1: test_hard")
plot_cm(out_e2_hard, "docs/week3/figures/confusion_e2_hard.png", "E2 (BERTurk): test_hard")
display(Image("docs/week3/figures/confusion_e1_hard.png"), Image("docs/week3/figures/confusion_e2_hard.png"))

**Ne oldu?** Karışıklık matrisinde satır = gerçek niyet, sütun = tahmin. Köşegen dışındaki sayılar hatalardır.
**Soru (kendine sor):** Hangi niyetler birbirine karışıyor? Örtük cümleler ("acıktım") hangi modelde daha iyi?

## 7. Sonuçları kaydet

In [ ]:
import json
os.makedirs("results", exist_ok=True)
cmp.to_csv("results/week3_comparison.csv")
hedef.to_csv("results/week3_targets.csv", index=False)
json.dump(rows, open("results/week3_metrics.json", "w"), indent=2, ensure_ascii=False)
e2_err.to_csv("results/errors_e2_test_hard.csv", index=False)
hatalar(out_e1_hard).to_csv("results/errors_e1_test_hard.csv", index=False)

with open("docs/week3/comparison.md", "w", encoding="utf-8") as f:
    f.write("# Hafta 3: E1 / E2 karşılaştırması\n\n")
    f.write("Colab'da çalıştırıldı (`notebooks/02_colab_week3.ipynb`).\n\n")
    f.write(cmp.to_markdown() + "\n\n## Hedefler (öneri formu)\n\n" + hedef.to_markdown(index=False) + "\n")

!zip -qr week3_sonuclar.zip results docs/week3 data/processed
print("hazır: week3_sonuclar.zip")

## 8. Sonuçları GitHub'a göndermek

`week3_sonuclar.zip` Colab'ın sol panelindeki **Dosyalar** simgesinde görünür. İki yol:

1. **Bana ver:** Zip'i indir, sohbete yükle. Ben repoya Hafta 3 sonucu olarak commit'leyeyim.
2. **Kendin yap:** Zip'i yerel repo klasörüne çıkar, `git add . && git commit -m "Hafta 3: Colab sonuçları" && git push`.

Not defterinin kendisini de (çıktılarıyla) kaydetmek için Colab'da *Dosya → GitHub'a kopya kaydet* seçeneğini kullanabilirsin.